# PadelVision — 00 Setup check (Kaggle / Colab)

Verifies that a fresh notebook can install the pipeline, download models, read your video and run the
person detector on the GPU. It also saves one frame as `court_frame.png`; you'll click the court corners on it in phase 1.

**Kaggle:** Settings → Accelerator **GPU T4 x2**, **Internet on**. Add your match video as a Dataset (Add Input).
**Colab:** Runtime → Change runtime type → **T4 GPU**. Put your video in Google Drive.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

Set the path to your match video:

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
from padelvision.video import probe
info = probe(VIDEO_PATH)
info.to_dict()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from padelvision.models import weights_path

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}  cuda available={torch.cuda.is_available()}  using device={DEVICE}")

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, info.frame_count // 2)
ok, frame = cap.read()
cap.release()
assert ok, "Could not read a frame"

cv2.imwrite(f"{WORK}/court_frame.png", frame)

model = YOLO(str(weights_path("person-detector")))
# Players are small in wide shots: a larger input size helps.
result = model.predict(frame, classes=[0], conf=0.3, imgsz=1280, device=DEVICE, verbose=False)[0]
print(f"people detected={len(result.boxes)}  inference={result.speed['inference']:.0f} ms")

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.axis("off");

**Expected:** a GPU device (`cuda:0`), the video's real fps and resolution, and boxes on the 4 players
(plus maybe spectators; phase 1 filters those out using the court). Download `court_frame.png` from the output panel.